# 🤰 Pregnancy RAG + Agents (Colab Setup)

This notebook sets up the Pregnancy RAG + Agent system in Google Colab: clones the repo, ingests the pregnancy guideline documents, generates embeddings, builds a ChromaDB vector index (using the same robustness fixes from the Hypertension embedding pipeline), evaluates retrieval quality, and launches the Streamlit UI.

> **Adapted from:** the embedding/ChromaDB architecture in `01_embed_and_store_chroma_FIXED.ipynb`, combined with the Colab setup + Streamlit/localtunnel launch flow from `Hypertension_Colab.ipynb`.
>
> **⚠️ Before running:** update the repo sub-folder, script names, data file names, and Streamlit filename (marked with `TODO`) to match wherever the pregnancy module actually lives in your repo — I mirrored the `hypertension_rag` folder/file naming convention since I don't have visibility into your repo's real pregnancy folder structure.

## 1. Clone the Repository

In [ ]:
# Import the `userdata` module to securely access your GitHub Token
from google.colab import userdata
import os

# Get the GH_TOKEN from Colab Secrets (if available)
try:
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    GH_TOKEN = None

repo_url = "https://github.com/Abdelrahmann-Mostafa/Pyramind---Hackathon.git"
if GH_TOKEN is not None:
    repo_url = repo_url.replace("https://", f"https://oauth2:{GH_TOKEN}@")

!git clone {repo_url}

# TODO: point this at the actual pregnancy module folder in the repo
%cd Pyramind---Hackathon/pregnancy_rag

print("Repository cloned successfully!")

## 2. Install Dependencies

In [ ]:
# Relax the anthropic pin the same way the hypertension setup does, then install requirements
!sed -i '/anthropic/d' requirements.txt
!pip install -r requirements.txt -q

# Explicit installs for the embedding / vector-store stack (matches the chroma notebook)
!pip install sentence-transformers chromadb pydantic torch PyMuPDF -q

import os
import sys
from pathlib import Path

PROJECT_ROOT = str(Path(os.getcwd()).parent)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

## 3. Setup Environment Variables
**IMPORTANT:** provide your Groq API key here (Colab Secrets is recommended).

In [ ]:
import os
from google.colab import userdata

# Put your API key in Colab Secrets under 'GROQ_API_KEY' or paste it below
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
except Exception:
    os.environ["GROQ_API_KEY"] = "your-api-key-here"  # REPLACE THIS IF NOT USING SECRETS

## 4. Day 1 — Document Ingestion & Embedding Generation

TODO: `run_ingestion_pipeline` / `run_embedding_pipeline` should read from wherever the pregnancy source documents live (e.g. `src/ingestion.py`, `src/embed_chunks.py` in the pregnancy module, or the shared versions if the repo reuses one pipeline for every module).

In [ ]:
from src.ingestion import run_ingestion_pipeline
from src.embed_chunks import run_embedding_pipeline

print('[Day 1] Running ingestion pipeline...')
chunks = run_ingestion_pipeline()
print(f'✅ Ingestion complete: {len(chunks)} chunks created')

print('[Day 1] Running embedding pipeline...')
embedded_chunks = run_embedding_pipeline()
print(f'✅ Embedding complete: {len(embedded_chunks)} chunks embedded')
print(f'   Embedding dimension: {embedded_chunks[0]["embedding_dim"]}')
print(f'   Model: {embedded_chunks[0]["embedding_model"]}')

## 5. Day 2 — Build the ChromaDB Vector Index

This reuses the **corrected** index-building logic from the chroma notebook, with three fixes baked in:
1. ✅ Query embedding mismatch fixed (manual encoding with the pritamdeka model)
2. ✅ ID matching logic corrected (`retrieved_ids = results['ids'][0]`)
3. ✅ Stale ChromaDB data cleared before recreation

Collection name is set to `pregnancy-guidelines` — rename if you'd prefer something else.

In [ ]:
from sentence_transformers import SentenceTransformer
import json

# Load the embedding model (must match the model used to embed the chunks above)
embedding_model = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO')
print('✅ Loaded embedding model: pritamdeka/S-PubMedBert-MS-MARCO (768-dim)')

In [ ]:
import json
import time
import chromadb
from pathlib import Path

def create_chroma_index_fixed(
    embeddings_file: str = "data/chunk_embeddings.json",
    chroma_path: str = "data/chroma_db",
    collection_name: str = "pregnancy-guidelines",
) -> chromadb.Collection:
    """
    Create the ChromaDB index with proper error handling (clears stale data,
    retries client init, and rebuilds the collection from scratch).
    """

    if not Path(embeddings_file).exists():
        raise FileNotFoundError(f"Embeddings file not found: {embeddings_file}")

    print(f"\n[ChromaDB] Loading embeddings...")
    with open(embeddings_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"  ✅ Loaded {len(chunks)} chunks")

    # Initialize with retry
    print(f"\n[ChromaDB] Initializing client...")
    for attempt in range(3):
        try:
            client = chromadb.PersistentClient(path=chroma_path)
            print(f"  ✅ Client ready")
            break
        except Exception as e:
            if attempt < 2:
                print(f"  ⚠️ Attempt {attempt+1} failed, retrying...")
                time.sleep(2)
            else:
                raise

    # Try to delete existing (stale) collection
    print(f"\n[ChromaDB] Handling existing collection...")
    try:
        client.get_collection(collection_name)
        client.delete_collection(collection_name)
        print(f"  ✅ Deleted existing collection")
        time.sleep(1)
    except ValueError:
        print(f"  ℹ️ No existing collection")
    except Exception as e:
        print(f"  ⚠️ Could not delete: {e}")

    # Create new collection
    print(f"\n[ChromaDB] Creating new collection...")
    try:
        collection = client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        print(f"  ✅ Collection created")
    except Exception as e:
        if "already exists" in str(e):
            collection = client.get_collection(collection_name)
            print(f"  ℹ️ Using existing collection")
        else:
            raise

    # Add data
    print(f"\n[ChromaDB] Adding {len(chunks)} vectors...")

    ids = [c["chunk_id"] for c in chunks]
    embeddings = [c["embedding"] for c in chunks]
    documents = [c["content"] for c in chunks]
    metadatas = []

    for c in chunks:
        meta = {k: v for k, v in c.items()
                if k not in ["embedding", "content", "embedding_model", "embedding_dim"]}
        for k, v in meta.items():
            if not isinstance(v, (str, int, float, bool)):
                meta[k] = str(v)
        metadatas.append(meta)

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas,
    )

    print(f"  ✅ Added {collection.count()} vectors")
    print(f"\n✅ ChromaDB index ready!")
    return collection


# Run it
print('[Day 2] Creating ChromaDB index...')
collection = create_chroma_index_fixed(
    embeddings_file='data/chunk_embeddings.json',
    chroma_path='data/chroma_db',
    collection_name='pregnancy-guidelines'
)
print(f'✅ ChromaDB index created with {collection.count()} vectors')

### Load Benchmark Queries
TODO: point at the pregnancy benchmark query file if it has a different name.

In [ ]:
with open('data/benchmark_queries.json', 'r', encoding='utf-8') as f:
    benchmark_queries = json.load(f)

print(f'✅ Loaded {len(benchmark_queries)} benchmark queries')
print(f'   Categories: {set(q.get("category", "unknown") for q in benchmark_queries)}')

### Test Retrieval With Correct Query Encoding

Queries must be encoded with the **same** model used for the chunks (pritamdeka) — ChromaDB's default embedding function would produce incompatible vectors.

In [ ]:
# Get a test query
test_query = benchmark_queries[0]['query']
print(f'Test Query: {test_query}')
print()

# Manually encode query with the pritamdeka model
query_vector = embedding_model.encode(test_query).tolist()
print(f'✅ Encoded query to {len(query_vector)}-dim vector using pritamdeka model')
print()

# Query ChromaDB with the proper embedding
results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

# Correct ID matching logic
retrieved_ids = results['ids'][0]
retrieved_dists = results['distances'][0]

print('Retrieved Top-3 Results:')
for rank, (chunk_id, distance) in enumerate(zip(retrieved_ids, retrieved_dists), 1):
    similarity = 1.0 - distance
    print(f'  [{rank}] {chunk_id}: similarity={similarity:.3f}')
print()

# Compare against ground truth
ground_truth = benchmark_queries[0]['ground_truth_chunk_ids']
hits = set(retrieved_ids).intersection(set(ground_truth))
print(f'Expected: {ground_truth}')
print(f'Matches: {hits}')
if hits:
    print('✅ Query retrieval is working correctly!')
else:
    print('⚠️ Check if ground truth IDs are in the index')

### Run Full Evaluation on All Benchmark Queries

In [ ]:
print('[Day 2] Running evaluation on all benchmark queries...')
print()

results_log = {
    'total_queries': len(benchmark_queries),
    'model': 'pritamdeka/S-PubMedBert-MS-MARCO',
    'k': 3,
    'compression': True,
    'queries_evaluated': []
}

total_precision = 0
total_recall = 0
total_mrr = 0

for i, query_item in enumerate(benchmark_queries, 1):
    query_text = query_item['query']
    ground_truth = set(query_item['ground_truth_chunk_ids'])

    query_vector = embedding_model.encode(query_text).tolist()

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=3
    )

    retrieved_ids = set(results['ids'][0])

    hits = retrieved_ids.intersection(ground_truth)
    precision = len(hits) / 3
    recall = len(hits) / len(ground_truth) if ground_truth else 0

    mrr = 0
    for rank, chunk_id in enumerate(results['ids'][0], 1):
        if chunk_id in ground_truth:
            mrr = 1.0 / rank
            break

    total_precision += precision
    total_recall += recall
    total_mrr += mrr

    query_result = {
        'query_id': query_item.get('query_id', f'q{i:03d}'),
        'precision': precision,
        'recall': recall,
        'mrr': mrr,
        'hits': len(hits)
    }
    results_log['queries_evaluated'].append(query_result)

    status = '✅' if hits else '❌'
    print(f'  [{i:2d}] {status} P:{precision:.2f} R:{recall:.2f} MRR:{mrr:.2f} | {query_item.get("query_id", f"q{i:03d}")}')

avg_precision = total_precision / len(benchmark_queries)
avg_recall = total_recall / len(benchmark_queries)
avg_mrr = total_mrr / len(benchmark_queries)

results_log['metrics'] = {
    'avg_precision': avg_precision,
    'avg_recall': avg_recall,
    'avg_mrr': avg_mrr
}

print()
print('='*60)
print('EVALUATION SUMMARY')
print('='*60)
print(f'Avg Precision: {avg_precision:.3f}')
print(f'Avg Recall:    {avg_recall:.3f}')
print(f'Avg MRR:       {avg_mrr:.3f}')

with open('data/retrieval_metrics_log.json', 'w', encoding='utf-8') as f:
    json.dump(results_log, f, indent=2)
print('\n✅ Metrics saved to data/retrieval_metrics_log.json')

## 6. Launch the Streamlit App

Same detached-start-then-tunnel fix used in the hypertension notebook: Streamlit is started in the background and we wait for it to report healthy **before** starting localtunnel, avoiding the 502 you'd get if the tunnel connects before the app is listening.

TODO: update `app_pregnancy.py` to match the actual Streamlit entrypoint filename for this module.

In [ ]:
import os, socket, subprocess, time, urllib.request

if not os.environ.get("GROQ_API_KEY"):
    raise SystemExit("GROQ_API_KEY not set - add it to Colab Secrets (key icon) and re-run this cell.")

print("==============================================================")
print("COPY THIS IP ADDRESS. You will need it for the localtunnel page:")
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip('\n'))
print("==============================================================")

def is_port_open(port, timeout=3.0):
    try:
        with socket.create_connection(('127.0.0.1', port), timeout=timeout):
            return True
    except OSError:
        return False

# TODO: swap app_pregnancy.py for the real Streamlit entrypoint if it's named differently
start_cmd = (
    'nohup streamlit run app_pregnancy.py '
    '--server.address 0.0.0.0 '
    '--server.port 8501 '
    '--server.headless true '
    '--server.enableCORS false '
    '--server.enableXsrfProtection false '
    '> streamlit.log 2>&1 & echo $!'
)
proc = subprocess.Popen(
    start_cmd,
    shell=True,
    executable='/bin/bash',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
pid = proc.communicate(timeout=15)[0].decode().strip()
print(f'[1/3] Streamlit launched (PID {pid or "?"}). Waiting for port 8501 ...')

deadline = time.time() + 120
while time.time() < deadline:
    if is_port_open(8501):
        print('[1/3] OK - Port 8501 is open, Streamlit process is alive.')
        break
    if pid and not os.path.exists(f'/proc/{pid}'):
        print('[1/3] ERROR - Streamlit exited early! Check streamlit.log below:\n')
        print(open('streamlit.log').read())
        raise SystemExit(1)
    time.sleep(2)
else:
    print('[1/3] ERROR - Port 8501 never opened. Check streamlit.log below:\n')
    print(open('streamlit.log').read())
    raise SystemExit(1)

print('[2/3] Waiting for Streamlit health endpoint (first-run model load can take a few min) ...')
healthy = False
for _ in range(120):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=3) as r:
            if r.read().decode().strip() == 'ok':
                healthy = True
                break
    except Exception:
        pass
    time.sleep(3)

if healthy:
    print('[2/3] OK - Streamlit is healthy and serving.')
else:
    print('[2/3] WARNING - Health endpoint not ready yet - app may still be loading heavy models.')
    print('     Tail streamlit.log for progress/errors:')
    print(open('streamlit.log').read()[-4000:])

print('[3/3] Starting localtunnel ...')
tunnel = subprocess.Popen(
    'npx localtunnel --port 8501',
    shell=True,
    executable='/bin/bash',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

for raw in iter(tunnel.stdout.readline, b''):
    line = raw.decode(errors='replace').strip()
    if line:
        print(line, flush=True)
    if 'loca.lt' in line:
        url = [tok for tok in line.split() if tok.startswith('https://')]
        if url:
            print('\nOPEN THIS IN YOUR BROWSER: ' + url[-1])
            print('(Enter your Colab IP when localtunnel asks for the tunnel password)')